In [1]:
import logging
from random import randint
import random
import errno
import functools
from loguru import logger
import signal
import time
import json
import faiss
import subprocess
import shutil
import psutil
import pickle
import heapq
from collections import defaultdict
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

import lean_dojo
from utils.lean_math_utils import *

In [15]:
#get_repo code

In [3]:
tokenizer_name = "morph-labs/morph-prover-v0-7b" #"internlm/internlm2-math-7b" #"ScalableMath/Lean-STaR-plus"  # 'Saisam/gpt-neo-math-small' #

tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
tokenizer.padding_side = "right"

# Define the separator token
sep_token = "<sep>"
pad_token = "<pad>"
eos_token = "<end>"

# Check if the separator token already exists in the vocabulary
if sep_token not in tokenizer.get_vocab():
    tokenizer.add_tokens([sep_token])
if pad_token not in tokenizer.get_vocab():
    tokenizer.add_tokens([pad_token])

# Set the separator token
tokenizer.sep_token = sep_token
tokenizer.pad_token = pad_token

tokenizer.add_special_tokens({
    'sep_token': sep_token,
    'pad_token': pad_token,
    'eos_token': eos_token
})

3

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM, AutoModelForSeq2SeqLM, AdamW
from tqdm import tqdm
import os

class InstructionResponseDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        instruction, response = self.data[idx]
        
        input_encoding = self.tokenizer(instruction, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")
        output_encoding = self.tokenizer(response, max_length=self.max_length, padding="max_length", truncation=True, return_tensors="pt")

        input_mask = input_encoding["attention_mask"].squeeze()

        return {
            "input_ids": input_encoding["input_ids"].squeeze(),
            "attention_mask": input_mask,
            "labels": output_encoding["input_ids"].squeeze()
        }

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

    return total_loss / len(dataloader)

# Initialize tokenizer and model
# model_name = "google/byt5-small"  # You can change this to any other Seq2Seq model
model_name = '/home/mcwave/code/automath/atp/models/flan_base/trained_model_178k'
# tokenizer = AutoTokenizer.from_pretrained("kaiyuy/leandojo-lean4-tacgen-byt5-small")

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
# model = torch.load(model_name)

# Set up optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/home/mcwave/anaconda3/envs/atp/lib/python3.10/site-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [6]:
# Test the model
def generate_response(instruction):
    input_ids = tokenizer(instruction, return_tensors="pt").input_ids.to(device)
#     input_ids = torch.cat((input_ids, torch.tensor([[28705]])), dim=1)
#     print(input_ids)
    output = model.generate(input_ids, max_length=200, num_beams=4, length_penalty=0, do_sample = True, eos_token_id = tokenizer.eos_token_id)
#     print(output)
#     output = model.generate(input_ids, max_length=200, num_beams=4, length_penalty=0)
#     return output
    return [tokenizer.decode(tac, skip_special_tokens=False) for tac in output]
def generate_tac(instruction):
    res = generate_response(instruction)
    return res[0].split("<s> ")[1].split("<pad>")[0]
test_instruction = """R : Type u_1\ninst✝ : Ring R\na : R\n⊢ a + 0 = a"""

response = generate_tac(test_instruction)
print(f"Instruction: {test_instruction}")
random.seed(42)
print(f"Response 1: {response}")

Instruction: R : Type u_1
inst✝ : Ring R
a : R
⊢ a + 0 = a
Response 1: rw [add_comm]


In [7]:
def get_proof(file_path, full_name):
    theorem = Theorem(repo, file_path, full_name)
    dojo, state_0 = Dojo(theorem).__enter__()
    curr_state = state_0
    pp_to_state = {}
    seen = set()
    beam = [(explore_state_complexity(curr_state.pp), curr_state.pp)]
    pp_to_state[curr_state.pp] = curr_state
    seen.add(curr_state.pp)
    beam_width = 2
    proven = False
    start_time = time.time()
    states_generated = []
    num_fails = 0
    while beam:
        new_beam = []
        
        for complexity, state in beam[:beam_width]:
            seen.add(state)
#             print("state", state)
#             print()
            state = pp_to_state[state]
            states_generated.append(time.time() - start_time)
            if type(state) == lean_dojo.interaction.dojo.ProofFinished:
                return True
            
            candidates = generate_response(state.pp)
            
            for tac in candidates:
                if time.time() - start_time > 120:
#                     print("Out of time")
                    return False
                new_state = dojo.run_tac(state, tac)
#                 print(tac)
                if type(new_state) in [LeanError,TimeoutError,TacticResult,DojoCrashError,DojoHardTimeoutError,DojoInitError,ProofGivenUp]:
#                     print("Error\n")
                    continue
                elif type(new_state) == lean_dojo.interaction.dojo.ProofFinished:
#                     print("Found proof\n")
                    return True, states_generated
                if new_state.pp in seen:
#                     print("Seen\n")
                    continue
#                 print("Successful\n")
                complexity = explore_state_complexity(new_state.pp, complexity)
                pp_to_state[new_state.pp] = new_state
                new_beam.append((complexity, new_state.pp))
        if not new_beam and num_fails < 10:
#             print("Failed beam search, trying again")
            num_fails += 1
            continue
        beam = heapq.nsmallest(beam_width, new_beam)
        
        if proven:
            return True, states_generated
        
        if not beam:
            print("Out of beams")
            return False, states_generated
    return False, states_generated

def explore_state_complexity(state, base_complexity=None, seen_target_freq=None):
    if '⊢ False' in state:
        return 1000000
    if base_complexity is not None:
        return base_complexity + 1
    complexity = 0
    lines = state.split('\n')
    targets = []
    min_freq = 1000000
    for line in lines:
        if line.startswith("⊢"):
            target = line[1:].strip()
            targets.append(target)
            if seen_target_freq is not None:
                if target in seen_target_freq:
                    min_freq = min(min_freq, seen_target_freq[target])
                    seen_target_freq[target] = seen_target_freq[target] + 1
                else:
                    min_freq = 0
                    seen_target_freq[target] = 1
    lengths = [len(x) for x in targets]
    complexity = max(lengths)/2 + sum(lengths)/2
    if seen_target_freq is not None:
        complexity += PENALTY_SEEN_TARGET_MULTIPLIER*min_freq
    return complexity

In [8]:
def get_proof_nobeam(file_path, full_name):
    theorem = Theorem(repo, file_path, full_name)
    dojo, state_0 = Dojo(theorem).__enter__()
    curr_state = state_0
    proven = False
    start_time = time.time()
    seen = set()
    num_tries = 0
    
    while num_tries < 10:
        if time.time() - start_time > 120:
            return False
        try:
            tac = generate_tac(curr_state.pp)
            new_state = dojo.run_tac(curr_state, tac)
        except:
            num_tries += 1
            continue
        if type(new_state) in [LeanError,TimeoutError,TacticResult,DojoCrashError,DojoHardTimeoutError,DojoInitError,ProofGivenUp]:
            num_tries += 1
            continue
        elif type(new_state) == lean_dojo.interaction.dojo.ProofFinished:
            return True
        
        if new_state.pp in seen:
            continue
        seen.add(new_state.pp)
        curr_state = new_state
        num_tries = 0
    
    return False

In [9]:
theorems_list = [("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.add_neg_cancel_right"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.add_left_cancel"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.add_right_cancel"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.zero_mul"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.neg_eq_of_add_eq_zero"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.eq_neg_of_add_eq_zero"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.neg_zero"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.neg_neg"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.self_sub"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.one_add_one_eq_two"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyRing.two_mul"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyGroup.mul_right_inv"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyGroup.mul_one"),
("MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean", "MyGroup.mul_inv_rev"),
("MIL/C02_Basics/S05_Proving_Facts_about_Algebraic_Structures.lean", "absorb1"),
("MIL/C02_Basics/S05_Proving_Facts_about_Algebraic_Structures.lean", "absorb2"),
("MIL/C02_Basics/S05_Proving_Facts_about_Algebraic_Structures.lean", "aux1"),
("MIL/C02_Basics/S05_Proving_Facts_about_Algebraic_Structures.lean", "aux2"),
("MIL/C02_Basics/S03_Using_Theorems_and_Lemmas.lean", "fact1"),
("MIL/C02_Basics/S03_Using_Theorems_and_Lemmas.lean", "fact2"),
("MIL/C02_Basics/S04_More_on_Order_and_Divisibility.lean", "C02S04.aux"),
("MIL/C03_Logic/S05_Disjunction.lean", "C03S05.MyAbs.le_abs_self"),
("MIL/C03_Logic/S05_Disjunction.lean", "C03S05.MyAbs.neg_le_abs_self"),
("MIL/C03_Logic/S05_Disjunction.lean", "C03S05.MyAbs.abs_add"),
("MIL/C03_Logic/S05_Disjunction.lean", "C03S05.MyAbs.lt_abs"),
("MIL/C03_Logic/S05_Disjunction.lean", "C03S05.MyAbs.abs_lt"),
("MIL/C03_Logic/S04_Conjunction_and_Iff.lean", "C03S04.aux"),
("MIL/C03_Logic/S04_Conjunction_and_Iff.lean", "C03S04.not_monotone_iff"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.my_lemma4"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.FnUb"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.FnLb"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.FnEven"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.FnOdd"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.Subset.trans"),
("MIL/C03_Logic/S01_Implication_and_the_Universal_Quantifier.lean", "C03S01.SetUb"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.ConvergesTo"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.convergesTo_const"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.convergesTo_add"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.convergesTo_mul_const"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.exists_abs_le_of_convergesTo"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.aux"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.convergesTo_mul"),
("MIL/C03_Logic/S06_Sequences_and_Convergence.lean", "C03S06.convergesTo_unique"),
("MIL/C03_Logic/S02_The_Existential_Quantifier.lean", "C03S02.FnUb"),
("MIL/C03_Logic/S02_The_Existential_Quantifier.lean", "C03S02.FnLb"),
("MIL/C03_Logic/S02_The_Existential_Quantifier.lean", "C03S02.FnHasUb"),
("MIL/C03_Logic/S02_The_Existential_Quantifier.lean", "C03S02.FnHasLb"),
("MIL/C03_Logic/S02_The_Existential_Quantifier.lean", "C03S02.fnUb_add"),
("MIL/C03_Logic/S03_Negation.lean", "C03S03.FnUb"),
("MIL/C03_Logic/S03_Negation.lean", "C03S03.FnLb"),
("MIL/C03_Logic/S03_Negation.lean", "C03S03.FnHasUb"),
("MIL/C03_Logic/S03_Negation.lean", "C03S03.FnHasLb"),
("MIL/C04_Sets_and_Functions/S02_Functions.lean", "inverse"),
("MIL/C04_Sets_and_Functions/S02_Functions.lean", "inverse_spec"),
("MIL/C04_Sets_and_Functions/S02_Functions.lean", "Cantor"),
("MIL/C04_Sets_and_Functions/S01_Sets.lean", "primes"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sbAux"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sbSet"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sbFun"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sb_right_inv"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sb_injective"),
("MIL/C04_Sets_and_Functions/S03_The_Schroeder_Bernstein_Theorem.lean", "sb_surjective"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "fac"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "pow_two_le_fac"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "sum_sqr"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.add"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.mul"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.zero_add"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.succ_add"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.add_comm"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.add_assoc"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.mul_add"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.zero_mul"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.succ_mul"),
("MIL/C05_Elementary_Number_Theory/S02_Induction_and_Recursion.lean", "MyNat.mul_comm"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.two_le"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.exists_prime_factor"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.primes_infinite"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "Nat.Prime.eq_of_dvd_of_prime"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.mem_of_dvd_prod_primes"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.primes_infinite'"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.bounded_of_ex_finset"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.ex_finset_of_bounded"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.mod_4_eq_3_or_mod_4_eq_3"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.two_le_of_mod_4_eq_3"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.aux"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.exists_prime_factor_mod_4_eq_3"),
("MIL/C05_Elementary_Number_Theory/S03_Infinitely_Many_Primes.lean", "C05S03.primes_mod_4_eq_3_infinite"),
("MIL/C05_Elementary_Number_Theory/S01_Irrational_Roots.lean", "even_of_even_sqr"),
("MIL/C05_Elementary_Number_Theory/S01_Irrational_Roots.lean", "factorization_mul'"),
("MIL/C05_Elementary_Number_Theory/S01_Irrational_Roots.lean", "factorization_pow'"),
("MIL/C05_Elementary_Number_Theory/S01_Irrational_Roots.lean", "Nat.Prime.factorization'"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.zero_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.one_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.add_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.neg_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.mul_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.zero_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.zero_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.one_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.one_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.add_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.add_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.neg_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.neg_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.mul_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.mul_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.instCommRing"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.sub_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.sub_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "Int.div'"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "Int.mod'"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "Int.div'_add_mod'"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "Int.abs_mod'_le"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "Int.mod'_eq"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "aux"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "sq_add_sq_eq_zero"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_nonneg"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_eq_zero"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_pos"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_mul"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.conj"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.conj_re"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.conj_im"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_conj"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.div_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.mod_def"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.norm_mod_lt"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.coe_natAbs_norm"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.natAbs_norm_mod_lt"),
("MIL/C06_Structures/S03_Building_the_Gaussian_Integers.lean", "gaussInt.not_norm_mul_left_lt_norm"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.AddGroup₁"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.Point"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.Point.add"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.Point.neg"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.Point.zero"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.Point.addGroupPoint"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.AddGroup₂"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.hasAddAddGroup₂"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.hasZeroAddGroup₂"),
("MIL/C06_Structures/S02_Algebraic_Structures.lean", "C06S02.hasNegAddGroup₂"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.Point"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.Point.add"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.Point.add_assoc"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.Point.smul"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.Point.smul_distrib"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.StandardTwoSimplex"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.StandardTwoSimplex.weightedAverage"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.StandardSimplex"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.StandardSimplex.midpoint"),
("MIL/C06_Structures/S01_Structures.lean", "C06S01.StandardSimplex.weightedAverage"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "Submonoid₁"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "SubMonoid₁Monoid"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "SubmonoidClass₁"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "Subgroup₁"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "SubgroupClass₁"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "Submonoid.Setoid"),
("MIL/C07_Hierarchies/S03_Subobjects.lean", "QuotientMonoid.mk"),
("MIL/C07_Hierarchies/S01_Basics.lean", "One₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "One₂"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Dia₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Semigroup₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Semigroup₂"),
("MIL/C07_Hierarchies/S01_Basics.lean", "DiaOneClass₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Monoid₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Monoid₂"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Inv₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Group₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "left_inv_eq_right_inv₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "inv_eq_of_dia"),
("MIL/C07_Hierarchies/S01_Basics.lean", "dia_inv"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddSemigroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Semigroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddMonoid₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Monoid₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "left_inv_eq_right_inv'"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddCommSemigroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "CommSemigroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddCommMonoid₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "CommMonoid₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddGroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Group₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "inv_eq_of_mul"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Group₃.mul_inv"),
("MIL/C07_Hierarchies/S01_Basics.lean", "mul_left_cancel₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "mul_right_cancel₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddCommGroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "CommGroup₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Ring₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "LE₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Preorder₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "PartialOrder₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "OrderedCommMonoid₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "SMul₃"),
("MIL/C07_Hierarchies/S01_Basics.lean", "Module₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "selfModule"),
("MIL/C07_Hierarchies/S01_Basics.lean", "nsmul₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "zsmul₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "abGrpModule"),
("MIL/C07_Hierarchies/S01_Basics.lean", "AddMonoid₄"),
("MIL/C07_Hierarchies/S01_Basics.lean", "mySMul"),
("MIL/C07_Hierarchies/S01_Basics.lean", "LT₁"),
("MIL/C07_Hierarchies/S01_Basics.lean", "PreOrder₂"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "isMonoidHom₁"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "isMonoidHom₂"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "MonoidHom₁"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "AddMonoidHom₁"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "RingHom₁"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "MonoidHomClass₁"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "badInst"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "MonoidHomClass₂"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "map_inv_of_inv"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "MonoidHomClass₃"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "OrderPresHom"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "OrderPresMonoidHom"),
("MIL/C07_Hierarchies/S02_Morphisms.lean", "OrderPresHomClass"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "conjugate"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "eq_bot_iff_card"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "inf_bot_of_coprime"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "conjugate_one"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "aux_card_eq"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "iso₁"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "iso₂"),
("MIL/C08_Groups_and_Rings/S01_Groups.lean", "finalIso"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseMap"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseMap_mk"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseMap_mk'"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseMap_inj"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "isCoprime_Inf"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseMap_surj"),
("MIL/C08_Groups_and_Rings/S02_Rings.lean", "chineseIso"),
("MIL/C09_Topology/S03_Topological_Spaces.lean", "aux"),
("MIL/C09_Topology/S01_Filters.lean", "Tendsto₁"),
("MIL/C09_Topology/S02_Metric_Spaces.lean", """cauchySeq_of_le_geometric_two'""")]
# theorems_list = [('MIL/C02_Basics/S02_Proving_Identities_in_Algebraic_Structures.lean', 'MyRing.neg_add_cancel_left'),]

In [16]:
# chapters = os.listdir("/home/mcwave/mathematics_in_lean/MIL")
total_attempts = 0
num_proven = 0
#500K: 39/118
for thm_path, thm_name in theorems_list[94:]:
    try:
        print(thm_path, thm_name)
        is_proven = get_proof_nobeam(thm_path, thm_name)
        num_proven += is_proven
        total_attempts += 1
        state_data = {"file_path": thm_path, "full_name": thm_name, "total_attempts": total_attempts, "is_proven": is_proven}
        with open("/home/mcwave/code/automath/atp/datasets/flant5_results_178k.json", "a") as outfile:
            json.dump(state_data, outfile)
            outfile.write("\n")
        print("res", num_proven, total_attempts)
    except Exception as e:
        print(e)
print(1/0)